# Train CNN (EfficientNet-B0) on 100,000 Bird Images
### Run this notebook on **Google Colab** (free GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mithunonline/EyeM/blob/master/notebooks/Train_CNN_Birds_Colab.ipynb)

**Dataset:** 525 Bird Species (~89,885 training images) from Kaggle  
**Model:** EfficientNet-B0 fine-tuned with transfer learning  
**Output:** `cnn_birds_finetuned.pth` saved to Google Drive

---
## Step 0: Before You Start
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Get your Kaggle API key: kaggle.com → Account → API → Create New Token → downloads `kaggle.json`
3. Run cells top to bottom

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Install / upgrade packages ────────────────────────────────────────────────
!pip install -q kaggle timm torchmetrics

## Step 1: Mount Google Drive (to save your trained model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/EyeM_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Models will be saved to: {SAVE_DIR}')

## Step 2: Upload Kaggle API Key

In [ ]:
# Upload your kaggle.json file
from google.colab import files
print('Upload your kaggle.json file (from kaggle.com → Account → API → Create New Token)')
uploaded = files.upload()

# Place it where kaggle CLI expects it
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API key configured.')

## Step 3: Download the 525 Bird Species Dataset (~1.3 GB)

In [ ]:
# Dataset: 525 Bird Species - Image Classification
# ~89,885 training images, 525 classes, ~1.3 GB
# https://www.kaggle.com/datasets/gpiosenka/100-bird-species

!kaggle datasets download -d gpiosenka/100-bird-species -p /content/birds --unzip
print('Download complete.')

In [ ]:
import os, pathlib

DATA_ROOT = '/content/birds'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VALID_DIR = os.path.join(DATA_ROOT, 'valid')
TEST_DIR  = os.path.join(DATA_ROOT, 'test')

# Count images
def count_images(directory):
    return sum(len(list(pathlib.Path(d).glob('*.jpg')) + 
                   list(pathlib.Path(d).glob('*.JPG')) +
                   list(pathlib.Path(d).glob('*.png')))
               for d in [os.path.join(directory, c) 
                         for c in os.listdir(directory)])

num_classes = len(os.listdir(TRAIN_DIR))
num_train   = count_images(TRAIN_DIR)
num_valid   = count_images(VALID_DIR)

print(f'Number of bird species (classes): {num_classes}')
print(f'Training images:                  {num_train:,}')
print(f'Validation images:                {num_valid:,}')
print(f'Sample classes: {os.listdir(TRAIN_DIR)[:5]}')

## Step 4: Data Augmentation & DataLoaders

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

BATCH_SIZE   = 64
NUM_WORKERS  = 2
IMAGE_SIZE   = 224
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# Training augmentations — make the model robust to real-world variation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),  # Random zoom/crop
    transforms.RandomHorizontalFlip(),                           # Mirror
    transforms.RandomRotation(15),                              # ±15° rotation
    transforms.ColorJitter(brightness=0.3, contrast=0.3,        # Color variation
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),                         # Occasional grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],            # ImageNet stats
                         std=[0.229, 0.224, 0.225]),
])

# Validation — no augmentation, just resize and normalize
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = ImageFolder(VALID_DIR, transform=val_transforms)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f'Training batches:   {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Classes:            {NUM_CLASSES}')
print(f'Images/batch:       {BATCH_SIZE}')

In [ ]:
# Visualize some training samples
import matplotlib.pyplot as plt
import numpy as np

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    img = img * STD + MEAN
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flat):
    if i < 18:
        ax.imshow(denormalize(images[i]))
        ax.set_title(CLASS_NAMES[labels[i].item()][:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Training Samples (with augmentation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5: Build Fine-tuned EfficientNet-B0 Model

In [ ]:
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights
import torch.nn as nn

# Load pretrained EfficientNet-B0 (trained on ImageNet-1k)
base_model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# ── Transfer learning strategy ─────────────────────────────────────────────
# Phase 1 (first 5 epochs): Freeze all backbone layers, only train the new head
#   → Fast convergence, new head learns to map features → bird species
# Phase 2 (remaining epochs): Unfreeze all layers, fine-tune the entire network
#   → Backbone adapts its features to bird-specific patterns

# Replace the classifier head
in_features = base_model.classifier[1].in_features  # 1280
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, 512),
    nn.SiLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, NUM_CLASSES),
)

# Freeze backbone initially
for param in base_model.features.parameters():
    param.requires_grad = False

model = base_model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Phase 1 — trainable params: {trainable:,} / {total:,}')
print(f'Output classes: {NUM_CLASSES} bird species')

## Step 6: Loss, Optimizer & Scheduler

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

NUM_EPOCHS      = 20
PHASE2_EPOCH    = 5    # After this epoch, unfreeze backbone
LR_HEAD         = 1e-3
LR_BACKBONE     = 1e-4

# Loss with label smoothing — prevents overconfidence
# Instead of hard 0/1 labels, uses 0.1/0.9
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Phase 1 optimizer — only classifier parameters
optimizer = optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=1e-4)

# OneCycleLR: ramp up LR then decay — finds better optima faster
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR_HEAD,
    steps_per_epoch=len(train_loader),
    epochs=PHASE2_EPOCH,
    pct_start=0.3,
)

print(f'Loss:       CrossEntropyLoss (label_smoothing=0.1)')
print(f'Optimizer:  AdamW')
print(f'Scheduler:  OneCycleLR')
print(f'Phase 1:    Epochs 1–{PHASE2_EPOCH} (head only, LR={LR_HEAD})')
print(f'Phase 2:    Epochs {PHASE2_EPOCH+1}–{NUM_EPOCHS} (full fine-tune, LR={LR_BACKBONE})')

## Step 7: Training Loop

In [ ]:
import time, json
from torch.cuda.amp import GradScaler, autocast  # Mixed precision training

scaler = GradScaler()  # Automatic mixed precision — ~2× faster on GPU

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc  = 0.0
best_model_path = os.path.join(SAVE_DIR, 'cnn_birds_best.pth')


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch_idx, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()
                with autocast():                    # float16 forward pass
                    outputs = model(images)         # (B, NUM_CLASSES)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()       # scaled backward
                scaler.unscale_(optimizer)          # unscale before clip
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)              # update weights
                scaler.update()
                scheduler.step()
            else:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

            if training and (batch_idx + 1) % 100 == 0:
                print(f'  Batch {batch_idx+1}/{len(loader)} | '
                      f'Loss: {total_loss/total:.4f} | '
                      f'Acc: {100.*correct/total:.2f}%')

    return total_loss / total, 100. * correct / total


print(f'Starting training for {NUM_EPOCHS} epochs...')
print('=' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    # ── Phase 2 transition: unfreeze backbone ─────────────────────────────
    if epoch == PHASE2_EPOCH + 1:
        print('\n>>> Phase 2: Unfreezing backbone for full fine-tuning...')
        for param in model.features.parameters():
            param.requires_grad = True

        # New optimizer with different LR for backbone vs head
        optimizer = optim.AdamW([
            {'params': model.features.parameters(), 'lr': LR_BACKBONE},
            {'params': model.classifier.parameters(), 'lr': LR_HEAD * 0.5},
        ], weight_decay=1e-4)

        scheduler = OneCycleLR(
            optimizer,
            max_lr=[LR_BACKBONE, LR_HEAD * 0.5],
            steps_per_epoch=len(train_loader),
            epochs=NUM_EPOCHS - PHASE2_EPOCH,
        )
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'    Trainable params now: {trainable:,}\n')

    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)

    elapsed = time.time() - start
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | '
          f'Time: {elapsed:.0f}s')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
            'num_classes': NUM_CLASSES,
            'architecture': 'efficientnet_b0',
        }, best_model_path)
        print(f'    ✓ Best model saved (val_acc={val_acc:.2f}%)')

print(f'\nTraining complete! Best val accuracy: {best_val_acc:.2f}%')
print(f'Model saved to: {best_model_path}')

## Step 8: Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_acc']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy', linewidth=2)
axes[0].plot(epochs_range, history['val_acc'],   'g-o', label='Val Accuracy',   linewidth=2)
axes[0].axvline(PHASE2_EPOCH, color='orange', linestyle='--', label='Phase 2 start')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('CNN (EfficientNet-B0) — Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[1].plot(epochs_range, history['val_loss'],   'g-o', label='Val Loss',   linewidth=2)
axes[1].axvline(PHASE2_EPOCH, color='orange', linestyle='--', label='Phase 2 start')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('CNN (EfficientNet-B0) — Loss', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cnn_birds_training_curves.png'), dpi=150)
plt.show()
print(f'Best Val Accuracy: {best_val_acc:.2f}%')

## Step 9: Evaluation on Test Set

In [ ]:
# Load best checkpoint
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} (val_acc={checkpoint["val_acc"]:.2f}%)')

# Test set evaluation
test_dataset = ImageFolder(TEST_DIR, transform=val_transforms)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100. * correct / total
print(f'Test Accuracy: {test_acc:.2f}% ({correct}/{total} correct)')

In [ ]:
# Per-class accuracy (top 20 and bottom 20)
import numpy as np
from collections import defaultdict

class_correct = defaultdict(int)
class_total   = defaultdict(int)
for pred, label in zip(all_preds, all_labels):
    class_total[label]   += 1
    class_correct[label] += int(pred == label)

class_acc = {CLASS_NAMES[k]: 100. * class_correct[k] / class_total[k]
             for k in class_total}
sorted_acc = sorted(class_acc.items(), key=lambda x: x[1], reverse=True)

print('Top 10 easiest bird species:')
for name, acc in sorted_acc[:10]:
    print(f'  {name:40s} {acc:.1f}%')

print('\nTop 10 hardest bird species:')
for name, acc in sorted_acc[-10:]:
    print(f'  {name:40s} {acc:.1f}%')

## Step 10: Export Model for Streamlit App

In [ ]:
# Save complete model package for the Streamlit app
export_path = os.path.join(SAVE_DIR, 'cnn_birds_finetuned.pth')

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'architecture': 'efficientnet_b0',
    'val_acc': best_val_acc,
    'test_acc': test_acc,
    'training_images': num_train,
}, export_path)

# Save class names as JSON (used by Streamlit)
json_path = os.path.join(SAVE_DIR, 'bird_class_names.json')
with open(json_path, 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print(f'Model exported to: {export_path}')
print(f'Class names saved: {json_path}')
print(f'\nFinal Results:')
print(f'  Classes:         {NUM_CLASSES} bird species')
print(f'  Training images: {num_train:,}')
print(f'  Best Val Acc:    {best_val_acc:.2f}%')
print(f'  Test Acc:        {test_acc:.2f}%')
print(f'\nDownload cnn_birds_finetuned.pth and bird_class_names.json')
print('and place them in the EyeM/models/ directory to use in the Streamlit app.')

## Step 11: Quick Inference Test

In [ ]:
import urllib.request, io
from PIL import Image

def predict_bird_cnn(image: Image.Image, top_k=5):
    model.eval()
    tensor = val_transforms(image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    top_idx = probs.argsort()[::-1][:top_k]
    return [(CLASS_NAMES[i], float(probs[i])) for i in top_idx]

# Test with a sample bird image
TEST_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/A_small_cup_of_coffee.JPG/640px-A_small_cup_of_coffee.JPG'
# Replace with a bird image URL to test
BIRD_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/4c/Starling_on_the_ground.jpg/640px-Starling_on_the_ground.jpg'

try:
    with urllib.request.urlopen(BIRD_URL) as r:
        img = Image.open(io.BytesIO(r.read())).convert('RGB')

    predictions = predict_bird_cnn(img, top_k=5)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].set_title('Test Bird Image'); axes[0].axis('off')

    labels_p = [p[0] for p in predictions]
    confs_p  = [p[1]*100 for p in predictions]
    axes[1].barh(labels_p[::-1], confs_p[::-1], color='#1565C0')
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('CNN Bird Predictions', fontweight='bold')
    plt.tight_layout(); plt.show()

    print('Predictions:')
    for lbl, prob in predictions:
        print(f'  {lbl:40s} {prob*100:.2f}%')
except Exception as e:
    print(f'Test failed: {e}. The model is ready — test with your own images.')